# Evaluacion de Modelos LSM (Lengua de Senas Mexicana)

Notebook de evaluacion que **carga el modelo entrenado y el test set** (sin re-entrenar) y produce todas las metricas para la defensa.

**Requisitos previos:**
- El modelo LSTM `model.keras` y `label_map.json` deben existir (carpeta `model/`).
- Los datos `.npy` deben estar en `model/data/<clase>/rep_*.npy`. En este repo aparecen como **borrados** en `git status`; si faltan, restauralos con:  `git restore model/data`  (o)  `git checkout -- model/data`.

**Notas importantes:**
- El Random Forest **no esta guardado** en el repo, asi que el notebook lo **entrena una vez y lo persiste** (`random_forest.joblib`); en ejecuciones posteriores lo lee.
- `train_model.py` **no guarda** el history de Keras, por lo que la curva de aprendizaje del LSTM solo esta disponible si reentrenas; se incluye la `learning_curve` de sklearn (RF) como equivalente.
- Las secciones 10 y 11 (mini-benchmarks ADR-09 / ADR-08) **entrenan** modelos: se ejecutan solo si las corres explicitamente.

## 0. Instalacion de dependencias
Ejecuta esta celda una sola vez. Si ya tienes el entorno, puedes saltarla.

In [ ]:
%pip install numpy pandas scikit-learn matplotlib seaborn tensorflow xgboost joblib

## 1. Imports y configuracion

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, learning_curve, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
import joblib

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Directorio de trabajo:', os.getcwd())

## 2. Localizar modelo y datos (autodescubrimiento)
Funciona aunque ejecutes el notebook desde la raiz del repo o desde dentro de `model/`.

In [ ]:
def first_existing(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None

MODEL_PATH = first_existing(['model.keras', os.path.join('model', 'model.keras'), os.path.join('..', 'model', 'model.keras')])
LABEL_MAP_PATH = first_existing(['label_map.json', os.path.join('model', 'label_map.json'), os.path.join('..', 'model', 'label_map.json')])
DATA_DIR = first_existing(['data', os.path.join('model', 'data'), os.path.join('..', 'model', 'data')])

print('MODEL_PATH     =', MODEL_PATH)
print('LABEL_MAP_PATH =', LABEL_MAP_PATH)
print('DATA_DIR       =', DATA_DIR)

## 3. Carga del dataset (test set)
Replica la logica de `train_model.py`: una subcarpeta por clase con secuencias `.npy`.

In [ ]:
# --- Mapa de clases: la FUENTE DE VERDAD es label_map.json (con el que se entreno el modelo) ---
# Acepta los dos formatos posibles del archivo: {"0": "femenino"} o {"femenino": 0}.
def load_label_names(path):
    if not path:
        return None
    with open(path, 'r', encoding='utf-8') as fh:
        raw = json.load(fh)

    def is_int(s):
        try:
            int(s); return True
        except (ValueError, TypeError):
            return False

    if all(is_int(k) for k in raw.keys()):      # {indice: nombre}
        pairs = {int(k): v for k, v in raw.items()}
    else:                                        # {nombre: indice}
        pairs = {int(v): k for k, v in raw.items()}
    return [pairs[i] for i in sorted(pairs)]

label_names = load_label_names(LABEL_MAP_PATH)
print('Clases segun label_map.json:', label_names)


# --- Carga del dataset ---
# Cargamos SOLO las clases que el modelo conoce (en el mismo orden de label_map.json) y
# con el shape que espera. Asi se ignoran carpetas sobrantes (p.ej. 'data/0/') que
# romperian la evaluacion del LSTM y el split estratificado.
EXPECTED_SHAPE = (90, 63)  # (frames, features) -- se valida contra cada .npy

def load_data(data_dir, allowed=None, expected_shape=None):
    if not data_dir or not os.path.isdir(data_dir):
        raise FileNotFoundError(
            'No se encontro la carpeta de datos. Los .npy aparecen como borrados en git status. '
            'Restauralos con:  git restore model/data')
    found = sorted([d for d in os.listdir(data_dir)
                    if os.path.isdir(os.path.join(data_dir, d))])
    classes = list(allowed) if allowed else found
    ignored = [d for d in found if d not in classes]
    if ignored:
        print('[aviso] Carpetas ignoradas (no estan en label_map.json):', ignored)

    name_to_idx = {c: i for i, c in enumerate(classes)}
    X, y, counts = [], [], {}
    for c in classes:
        cdir = os.path.join(data_dir, c)
        if not os.path.isdir(cdir):
            print('[aviso] Falta la carpeta de la clase:', c); counts[c] = 0; continue
        kept = 0
        for f in sorted(fn for fn in os.listdir(cdir) if fn.endswith('.npy')):
            arr = np.load(os.path.join(cdir, f))
            if expected_shape and tuple(arr.shape) != tuple(expected_shape):
                continue   # se ignora cualquier .npy con shape inesperado
            X.append(arr); y.append(name_to_idx[c]); kept += 1
        counts[c] = kept
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64), classes, counts

X, y, class_names, counts_per_class = load_data(
    DATA_DIR, allowed=label_names, expected_shape=EXPECTED_SHAPE)
num_classes = len(class_names)

print('Shape de X (muestras, frames, features):', X.shape)
print('Shape de y                            :', y.shape)
print('Clases (indice -> nombre)             :', dict(enumerate(class_names)))
print('Conteo por clase                      :', counts_per_class)

In [ ]:
# Confirmacion: las clases cargadas coinciden con label_map.json del modelo.
print('Nombres de clase (indice -> nombre):')
for i, n in enumerate(class_names):
    print('  ', i, '->', n)

if label_names is not None and list(class_names) != list(label_names):
    print('\n[AVISO] El orden de clases NO coincide con label_map.json; revisa las carpetas en data/.')
else:
    print('\nOK: el orden coincide con label_map.json (consistente con la salida del modelo).')

## 4. Split train / val / test
Reproduce `train_model.py`: `train_test_split(test_size=0.2, random_state=42, stratify=y)` y, dentro de `fit`, `validation_split=0.2` sobre el set de entrenamiento.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

n_total = len(X)
n_test = len(X_test)
n_val = int(round(0.2 * len(X_train)))
n_train = len(X_train) - n_val

print('Total :', n_total)
print('Train : {} ({:.0%})'.format(n_train, n_train / n_total))
print('Val   : {} ({:.0%})'.format(n_val, n_val / n_total))
print('Test  : {} ({:.0%})'.format(n_test, n_test / n_total))

## 5. Balance de clases (numero de muestras por clase)

In [ ]:
uniq, cnt = np.unique(y, return_counts=True)
names = [class_names[i] for i in uniq]
dist = pd.DataFrame({'clase': names, 'muestras': cnt})
display(dist)

plt.figure(figsize=(8, 4))
sns.barplot(data=dist, x='clase', y='muestras', color='steelblue')
plt.title('Numero de muestras por clase (balance del dataset)')
plt.xlabel('Clase'); plt.ylabel('Muestras')
plt.tight_layout(); plt.show()

print('Balance min/max:', int(cnt.min()), '/', int(cnt.max()),
      '-> ratio', round(cnt.max() / max(cnt.min(), 1), 2))

## 6. Dimension de features y ventana temporal

In [ ]:
n_frames = X.shape[1]
n_feat_per_frame = X.shape[2]
print('Ventana temporal (frames por secuencia):', n_frames)
print('Features por frame                      :', n_feat_per_frame)
print('Entrada del LSTM (frames, features)     :', (n_frames, n_feat_per_frame))
print('Features del Random Forest (mean-pool)  :', n_feat_per_frame)
print('  -> 63 = 21 landmarks de mano x 3 (x,y,z); 126 = 2 manos; etc.')

## 7. Cargar el LSTM entrenado y su arquitectura

In [ ]:
import tensorflow as tf
from tensorflow import keras

try:
    lstm = keras.models.load_model(MODEL_PATH)
    lstm.summary()
except Exception as e:
    print('No se pudo cargar el modelo. Revisa la version de TensorFlow/Keras.')
    print('Error:', e)
    raise

In [ ]:
# Hiperparametros / configuracion del LSTM
print('Optimizador:', lstm.optimizer.__class__.__name__)
try:
    print('Learning rate:', float(lstm.optimizer.learning_rate.numpy()))
except Exception:
    pass
print('Loss       :', lstm.loss)
print()
print('Capas (tipo -> {unidades / dropout / activacion}):')
for layer in lstm.layers:
    lc = layer.get_config()
    info = {k: lc.get(k) for k in ('units', 'rate', 'activation') if k in lc}
    print('  -', layer.__class__.__name__, info)

## 8. Evaluacion del LSTM en test
Accuracy global, classification_report (precision/recall/F1 por clase), matriz de confusion y pares mas confundidos.

In [ ]:
proba = lstm.predict(X_test, verbose=0)
y_pred = np.argmax(proba, axis=1)

acc = accuracy_score(y_test, y_pred)
print('Accuracy global LSTM (test): {:.4f}  ({:.2%})'.format(acc, acc))
print()
print(classification_report(y_test, y_pred, labels=list(range(num_classes)),
                            target_names=class_names, zero_division=0))

In [ ]:
def plot_cm(y_true, y_hat, title):
    cm = confusion_matrix(y_true, y_hat, labels=list(range(num_classes)))
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title); plt.xlabel('Prediccion'); plt.ylabel('Real')
    plt.tight_layout(); plt.show()
    return cm

cm_lstm = plot_cm(y_test, y_pred, 'Matriz de confusion - LSTM (test)')

In [ ]:
def most_confused(cm, top=10):
    pairs = []
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            if i != j and cm[i, j] > 0:
                pairs.append((class_names[i], class_names[j], int(cm[i, j])))
    pairs.sort(key=lambda t: t[2], reverse=True)
    return pd.DataFrame(pairs[:top], columns=['real', 'predicha', 'veces'])

print('Pares de clases mas confundidos (LSTM):')
display(most_confused(cm_lstm))

## 9. Random Forest: entrenar/cargar y analizar
El RF no maneja secuencias: agregamos la dimension temporal con **mean-pooling** -> `(muestras, features)`. Si `features == 63`, son los 63 del RF. No hay RF guardado en el repo, asi que se entrena una vez y se persiste en `random_forest.joblib`.

In [ ]:
X_rf = X.mean(axis=1)  # mean-pool temporal -> (muestras, features)
Xtr_rf, Xte_rf, ytr_rf, yte_rf = train_test_split(
    X_rf, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

RF_PATH = 'random_forest.joblib'
if os.path.exists(RF_PATH):
    rf = joblib.load(RF_PATH)
    print('RF cargado de', RF_PATH)
else:
    rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(Xtr_rf, ytr_rf)
    joblib.dump(rf, RF_PATH)
    print('RF entrenado y guardado en', RF_PATH)

print('Features de entrada del RF:', X_rf.shape[1])

In [ ]:
# Hiperparametros del Random Forest
params = rf.get_params()
for k in ['n_estimators', 'max_depth', 'criterion', 'max_features',
          'min_samples_split', 'min_samples_leaf', 'bootstrap']:
    print('{:18s}: {}'.format(k, params.get(k)))

In [ ]:
yrf_pred = rf.predict(Xte_rf)
acc_rf = accuracy_score(yte_rf, yrf_pred)
print('Accuracy global RF (test): {:.4f}  ({:.2%})'.format(acc_rf, acc_rf))
print()
print(classification_report(yte_rf, yrf_pred, labels=list(range(num_classes)),
                            target_names=class_names, zero_division=0))
cm_rf = plot_cm(yte_rf, yrf_pred, 'Matriz de confusion - Random Forest (test)')
print('Pares mas confundidos (RF):')
display(most_confused(cm_rf))

## 10. Curvas de aprendizaje
- **LSTM:** solo disponible si existe un history guardado (train_model.py no lo persiste).
- **RF:** `learning_curve` de sklearn como equivalente (no tiene epocas).

In [ ]:
# Curva del LSTM si se guardo el history (de lo contrario, se informa).
hist_path = first_existing(['history.json', 'lstm_history.json'])
if hist_path:
    with open(hist_path, 'r', encoding='utf-8') as fh:
        hist = json.load(fh)
    h = hist.get('history', hist)
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(h.get('loss', []), label='train')
    plt.plot(h.get('val_loss', []), label='val')
    plt.title('LSTM - loss'); plt.xlabel('epoca'); plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(h.get('accuracy', []), label='train')
    plt.plot(h.get('val_accuracy', []), label='val')
    plt.title('LSTM - accuracy'); plt.xlabel('epoca'); plt.legend()
    plt.tight_layout(); plt.show()
else:
    print('No hay history del LSTM guardado (train_model.py no lo persiste).')
    print('Opciones: (a) reentrenar guardando history.json; (b) usar la learning_curve del RF (abajo).')

In [ ]:
# Curva de aprendizaje del RF (equivalente sklearn)
counts_min = int(np.min(np.bincount(y)))
nsplit = max(2, min(5, counts_min))
cv = StratifiedKFold(n_splits=nsplit, shuffle=True, random_state=RANDOM_STATE)
sizes, train_sc, val_sc = learning_curve(
    RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    X_rf, y, cv=cv, train_sizes=np.linspace(0.2, 1.0, 5),
    scoring='accuracy', n_jobs=-1)

plt.figure(figsize=(7, 4))
plt.plot(sizes, train_sc.mean(axis=1), 'o-', label='train')
plt.plot(sizes, val_sc.mean(axis=1), 'o-', label='cv')
plt.title('Curva de aprendizaje - Random Forest (sklearn)')
plt.xlabel('Muestras de entrenamiento'); plt.ylabel('Accuracy')
plt.legend(); plt.tight_layout(); plt.show()

## 11. Mini-benchmark ADR-09: RF vs SVM vs XGBoost vs MLP
Mismos datos tabulares (mean-pool). Respalda la decision del Random Forest con numeros (accuracy / F1-macro).

In [ ]:
def evaluate(name, clf, Xtr, ytr, Xte, yte):
    clf.fit(Xtr, ytr)
    p = clf.predict(Xte)
    return {'modelo': name,
            'accuracy': accuracy_score(yte, p),
            'f1_macro': f1_score(yte, p, average='macro', zero_division=0)}

models = {
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    'SVM (RBF)': SVC(kernel='rbf', C=10, gamma='scale', random_state=RANDOM_STATE),
    'MLP': MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=1000, random_state=RANDOM_STATE),
}
try:
    from xgboost import XGBClassifier
    models['XGBoost'] = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                                      subsample=0.9, eval_metric='mlogloss', random_state=RANDOM_STATE)
except Exception as e:
    print('XGBoost no disponible:', e)

results = [evaluate(n, c, Xtr_rf, ytr_rf, Xte_rf, yte_rf) for n, c in models.items()]
bench = pd.DataFrame(results).sort_values('f1_macro', ascending=False).reset_index(drop=True)
display(bench)

In [ ]:
bm = bench.melt(id_vars='modelo', value_vars=['accuracy', 'f1_macro'],
                var_name='metrica', value_name='valor')
plt.figure(figsize=(8, 4))
sns.barplot(data=bm, x='modelo', y='valor', hue='metrica')
plt.title('ADR-09: comparativa de clasificadores tabulares')
plt.ylim(0, 1); plt.tight_layout(); plt.show()

## 12. Mini-benchmark ADR-08: LSTM vs GRU vs 1D-CNN
Mismas secuencias `(frames, features)` y mismo split. **Entrena 3 modelos** (puede tardar).

In [ ]:
from tensorflow.keras import layers

def make_lstm(inp, ncl):
    return keras.Sequential([
        layers.Input(shape=inp),
        layers.LSTM(64, return_sequences=True), layers.Dropout(0.3),
        layers.LSTM(64), layers.Dropout(0.3),
        layers.Dense(64, activation='relu'), layers.Dropout(0.3),
        layers.Dense(ncl, activation='softmax')])

def make_gru(inp, ncl):
    return keras.Sequential([
        layers.Input(shape=inp),
        layers.GRU(64, return_sequences=True), layers.Dropout(0.3),
        layers.GRU(64), layers.Dropout(0.3),
        layers.Dense(64, activation='relu'), layers.Dropout(0.3),
        layers.Dense(ncl, activation='softmax')])

def make_cnn(inp, ncl):
    return keras.Sequential([
        layers.Input(shape=inp),
        layers.Conv1D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling1D(2),
        layers.Conv1D(128, 3, activation='relu', padding='same'),
        layers.GlobalAveragePooling1D(),
        layers.Dense(64, activation='relu'), layers.Dropout(0.3),
        layers.Dense(ncl, activation='softmax')])

inp = (n_frames, n_feat_per_frame)
builders = {'LSTM': make_lstm, 'GRU': make_gru, '1D-CNN': make_cnn}
seq_results = []
for name, build in builders.items():
    m = build(inp, num_classes)
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
    m.fit(X_train, y_train, validation_split=0.2, epochs=80, batch_size=16, verbose=0, callbacks=[es])
    p = np.argmax(m.predict(X_test, verbose=0), axis=1)
    seq_results.append({'modelo': name,
                        'accuracy': accuracy_score(y_test, p),
                        'f1_macro': f1_score(y_test, p, average='macro', zero_division=0)})
    print(name, 'listo')

seq_bench = pd.DataFrame(seq_results).sort_values('f1_macro', ascending=False).reset_index(drop=True)
display(seq_bench)

In [ ]:
sm = seq_bench.melt(id_vars='modelo', value_vars=['accuracy', 'f1_macro'],
                    var_name='metrica', value_name='valor')
plt.figure(figsize=(7, 4))
sns.barplot(data=sm, x='modelo', y='valor', hue='metrica')
plt.title('ADR-08: LSTM vs GRU vs 1D-CNN (mismas secuencias)')
plt.ylim(0, 1); plt.tight_layout(); plt.show()

## 13. Resumen para la defensa

In [ ]:
print('=' * 50)
print('RESUMEN')
print('=' * 50)
print('Clases               :', num_classes, class_names)
print('Total muestras       :', n_total)
print('Split train/val/test :', n_train, '/', n_val, '/', n_test)
print('Entrada LSTM         :', (n_frames, n_feat_per_frame))
print('Features RF          :', X_rf.shape[1])
print('Accuracy LSTM (test) : {:.2%}'.format(acc))
print('Accuracy RF (test)   : {:.2%}'.format(acc_rf))
print()
print('ADR-09 (clasificadores tabulares):')
display(bench)
print('ADR-08 (modelos de secuencia):')
display(seq_bench)